In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
spark

In [4]:
from pyspark.sql import functions as F
from pyspark.sql import Row

# Simulating raw JSON payloads landing in Bronze table
data = [
    Row(payload='{"event_id": "e001", "event_time": "2025-08-04T10:15:00Z", "user": {"customer_id": "c105", "device": "iOS"}, "items": [{"product_id": "p99", "price": 29.99}, {"product_id": "p42", "price": 15.50}]}'),
    Row(payload='{"event_id": "e002", "event_time": "2025-08-04T10:16:30Z", "user": {"customer_id": "c210", "device": "Android"}, "items": [{"product_id": "p99", "price": 29.99}]}'),
    Row(payload='{"event_id": "e003", "event_time": "2025-08-04T10:18:00Z", "user": {"customer_id": "c008", "device": "Web"}, "items": []}') # User bought nothing, just an event
]

bronze_df = spark.createDataFrame(data)
bronze_df.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|payload                                                                                                                                                                                              |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"event_id": "e001", "event_time": "2025-08-04T10:15:00Z", "user": {"customer_id": "c105", "device": "iOS"}, "items": [{"product_id": "p99", "price": 29.99}, {"product_id": "p42", "price": 15.50}]}|
|{"event_id": "e002", "event_time": "2025-08-04T10:16:30Z", "user": {"customer_id": "c210", "device": "Android"}, "items": [{"product_id": "p99", "price": 29.99}]}                                   |


In [5]:
bronze_df.printSchema()

root
 |-- payload: string (nullable = true)



In [6]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType, DoubleType

schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_time", TimestampType()),
    StructField("user", StructType([
        StructField("customer_id", StringType()),
        StructField("device", StringType())
    ])),
    StructField("items", ArrayType(
    StructType([
        StructField("product_id", StringType()),
        StructField("price", DoubleType())
    ])
))
])

In [10]:
from pyspark.sql import functions as F
parsed_df = bronze_df.withColumn("parsed", F.from_json(F.col("payload"), schema)).drop("payload")

In [13]:
parsed_df.show(truncate=False)

+---------------------------------------------------------------------+
|parsed                                                               |
+---------------------------------------------------------------------+
|{e001, 2025-08-04 06:15:00, {c105, iOS}, [{p99, 29.99}, {p42, 15.5}]}|
|{e002, 2025-08-04 06:16:30, {c210, Android}, [{p99, 29.99}]}         |
|{e003, 2025-08-04 06:18:00, {c008, Web}, []}                         |
+---------------------------------------------------------------------+



In [14]:
parsed_df.printSchema()

root
 |-- parsed: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_time: timestamp (nullable = true)
 |    |-- user: struct (nullable = true)
 |    |    |-- customer_id: string (nullable = true)
 |    |    |-- device: string (nullable = true)
 |    |-- items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- product_id: string (nullable = true)
 |    |    |    |-- price: double (nullable = true)



In [15]:
# from_json() Takes a Column, Not a DataFrame
# from_json() is a column-level function. It takes a single column (F.col("payload")), not the whole DataFrame. It returns a new column of struct type

exploded_df = parsed_df.withColumn("items", F.explode_outer(F.col("parsed.items")))
exploded_df.show(truncate=False)

+---------------------------------------------------------------------+------------+
|parsed                                                               |items       |
+---------------------------------------------------------------------+------------+
|{e001, 2025-08-04 06:15:00, {c105, iOS}, [{p99, 29.99}, {p42, 15.5}]}|{p99, 29.99}|
|{e001, 2025-08-04 06:15:00, {c105, iOS}, [{p99, 29.99}, {p42, 15.5}]}|{p42, 15.5} |
|{e002, 2025-08-04 06:16:30, {c210, Android}, [{p99, 29.99}]}         |{p99, 29.99}|
|{e003, 2025-08-04 06:18:00, {c008, Web}, []}                         |null        |
+---------------------------------------------------------------------+------------+



In [21]:
silver_df = exploded_df.select(
    F.col("parsed.event_id").alias("event_id"),
    F.col("parsed.event_time").alias("event_time"),  
    F.col("parsed.user.customer_id").alias("customer_id"),             # reach inside user struct
    F.col("parsed.user.device").alias("device"),
    F.col("items.product_id").alias("product_id"),                      # reach inside exploded item
    F.col("items.price").alias("price"))

In [22]:
silver_df.show(truncate=False)

+--------+-------------------+-----------+-------+----------+-----+
|event_id|event_time         |customer_id|device |product_id|price|
+--------+-------------------+-----------+-------+----------+-----+
|e001    |2025-08-04 06:15:00|c105       |iOS    |p99       |29.99|
|e001    |2025-08-04 06:15:00|c105       |iOS    |p42       |15.5 |
|e002    |2025-08-04 06:16:30|c210       |Android|p99       |29.99|
|e003    |2025-08-04 06:18:00|c008       |Web    |null      |null |
+--------+-------------------+-----------+-------+----------+-----+



In [23]:
silver_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)

